# Strike Classifier Training

Takes labeled landmark windows from the app's StrikeRecorder and trains a Conv1D + LSTM classifier.
Exports a TF.js model you drop into `public/models/strike-classifier/`.

---
## How to use this notebook
1. Run the **Setup** cell to install dependencies.
2. Upload your `strike-training-data-*.json` file when prompted.
3. Run all cells to train and export the model.
4. Download the resulting `strike-classifier.zip` and extract into `public/models/` in your app.

In [ ]:
# @title Setup
import json
import math
import random
import zipfile
import os
from io import StringIO
from google.colab import files

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

# Install TF.js converter
!pip install -q tensorflowjs
import tensorflowjs as tfjs

In [ ]:
# @title Upload your training data
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'Loaded: {filename} ({len(uploaded[filename])} bytes)')

with open(filename) as f:
  raw = json.load(f)

print(f'Samples: {len(raw)}')
strike_types = list(set(s['strike'] for s in raw))
print(f'Strike types: {strike_types}')

for st in strike_types:
  count = sum(1 for s in raw if s['strike'] == st)
  print(f'  {st}: {count} samples')

WINDOW_SIZE = raw[0]['metadata']['windowSize']
FEATURES_PER_FRAME = len(raw[0]['frames'][0])
print(f'\nWindow size: {WINDOW_SIZE} frames')
print(f'Features per frame: {FEATURES_PER_FRAME} (33 landmarks x 4)')

## Preprocessing

Raw MediaPipe landmarks are in image coordinates. We need to make them invariant to:
- Camera distance (scale by torso height)
- Position in frame (center at hip midpoint)

Each frame becomes `33 × 3 = 99` features (dropping visibility, which the model shouldn't need).

In [ ]:
# @title Preprocess landmarks
# Landmark indices (MediaPipe Pose)
LEFT_HIP = 23
RIGHT_HIP = 24
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
NOSE = 0

def normalize_frame(frame):
  """Convert raw [x,y,z,v] * 33 to centered + scaled (x,y,z) * 33."""
  arr = np.array(frame).reshape(-1, 4)  # (33, 4)
  xyz = arr[:, :3]  # (33, 3) — keep x, y, z only

  # Center at hip midpoint
  hip_mid = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
  centered = xyz - hip_mid

  # Scale by torso height (shoulder midpoint → hip midpoint distance)
  shoulder_mid = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
  torso_height = np.linalg.norm(shoulder_mid - hip_mid)
  if torso_height < 1e-6:
    torso_height = 1.0
  scaled = centered / torso_height

  return scaled.flatten().astype(np.float32)  # (99,)


def sample_to_sequence(sample):
  """Convert a StrikeSample dict to (normalized_frames, label_index)."""
  frames = [normalize_frame(f) for f in sample['frames']]
  return np.stack(frames, axis=0)  # (WINDOW_SIZE, 99)


# Build dataset
strike_to_idx = {s: i for i, s in enumerate(sorted(strike_types))}
idx_to_strike = {i: s for s, i in strike_to_idx.items()}
NUM_CLASSES = len(strike_types)
print(f'Classes: {idx_to_strike}')

X_list, y_list = [], []
for s in raw:
  seq = sample_to_sequence(s)
  X_list.append(seq)
  y_list.append(strike_to_idx[s['strike']])

X = np.stack(X_list, axis=0)  # (N, WINDOW_SIZE, 99)
y = keras.utils.to_categorical(y_list, num_classes=NUM_CLASSES)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'X range: [{X.min():.3f}, {X.max():.3f}]')

## Model Architecture

A lightweight Conv1D stack suitable for browser inference.

In [ ]:
# @title Build model
def build_model(window_size, num_features, num_classes):
  inputs = keras.Input(shape=(window_size, num_features))

  x = layers.Conv1D(64, 5, activation='relu', padding='same')(inputs)
  x = layers.BatchNormalization()(x)
  x = layers.MaxPooling1D(2)(x)

  x = layers.Conv1D(128, 5, activation='relu', padding='same')(x)
  x = layers.BatchNormalization()(x)
  x = layers.MaxPooling1D(2)(x)

  x = layers.Conv1D(256, 3, activation='relu', padding='same')(x)
  x = layers.BatchNormalization()(x)
  x = layers.GlobalAveragePooling1D()(x)

  x = layers.Dense(128, activation='relu')(x)
  x = layers.Dropout(0.4)(x)
  x = layers.Dense(num_classes, activation='softmax')(x)

  model = keras.Model(inputs=inputs, outputs=x)
  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=0.001),
      loss='categorical_crossentropy',
      metrics=['accuracy'],
  )
  return model


model = build_model(WINDOW_SIZE, 99, NUM_CLASSES)
model.summary()

In [ ]:
# @title Split and train
# Shuffle
indices = np.random.permutation(len(X))
X_shuf = X[indices]
y_shuf = y[indices]

# Split (80/20 or use all for training if very few samples)
split = max(1, int(len(X) * 0.2))
if split < 2:
  # Too few samples — train on all, skip validation
  X_train, y_train = X_shuf, y_shuf
  X_val, y_val = X_shuf[:1], y_shuf[:1]
  print(f'Only {len(X)} samples — training on all, no real validation')
else:
  X_train, X_val = X_shuf[:-split], X_shuf[-split:]
  y_train, y_val = y_shuf[:-split], y_shuf[-split:]
  print(f'Train: {len(X_train)}, Val: {len(X_val)}')

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val) if len(X_val) > 1 else None,
    epochs=200,
    batch_size=8,
    verbose=2,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss' if len(X_val) > 1 else 'loss',
            patience=30,
            restore_best_weights=True,
        ),
    ],
)

In [ ]:
# @title Evaluate
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='train')
if 'val_loss' in history.history:
  ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history.history['accuracy'], label='train')
if 'val_accuracy' in history.history:
  ax2.plot(history.history['val_accuracy'], label='val')
ax2.set_title('Accuracy')
ax2.legend()
plt.show()

In [ ]:
# @title Test on a few samples
predictions = model.predict(X[:min(10, len(X))])
for i in range(min(5, len(X))):
  pred_idx = np.argmax(predictions[i])
  true_idx = np.argmax(y[i])
  print(f'Sample {i}: true={idx_to_strike[true_idx]}, pred={idx_to_strike[pred_idx]} ({predictions[i][pred_idx]:.1%})')

In [ ]:
# @title Export to TF.js
export_dir = 'strike-classifier'
if os.path.exists(export_dir):
  import shutil
  shutil.rmtree(export_dir)

tfjs.converters.save_keras_model(model, export_dir)
print(f'Exported to {export_dir}/')
print(f'Files: {os.listdir(export_dir)}')

In [ ]:
# @title Download model
!zip -r strike-classifier.zip strike-classifier/
files.download('strike-classifier.zip')
print('Downloaded strike-classifier.zip')

## Next steps in the app

1. **Extract** `strike-classifier.zip` into `public/models/strike-classifier/`
2. **Install** TensorFlow.js in the app:
   ```bash
   npm install @tensorflow/tfjs @tensorflow/tfjs-backend-webgl
   ```
3. **Create** `src/analysis/strikeDetection.ts` to load the model and run inference
4. **Wire** it into `CameraView.tsx` — feed normalized landmarks into the model every ~300ms
5. **Display** results in the overlay

---
## Tips for better results

- **Collect more data**: 20+ samples per strike type dramatically improves accuracy.
- **Augment**: The notebook doesn't include augmentation yet (jitter, scaling, time-warp). Add it before training for better generalization.
- **Balanced classes**: Collect roughly equal samples per strike type.
- **Background class**: Add a "no strike" / "idle" class — record windows where you're standing still.
- **Diverse angles**: Record from different camera positions and body orientations.